Ya vimos en el archivo de exploración que el target cnt es la suma de casual y registered, y cada una de estas categorías tiene una dinámica diferente. Por ejemplo, vimos que los usuarios registered son en general muchos más que los casual. Por eso, haremos un modelo de regresión para predecir la cantidad de usuarios registered por día y otro para los usuarios casual; la predicción de cnt será la suma de las dos anteriores.

Vemos que todos los atributos tienen valores numéricos excepto la fecha, que vuelve a ser representada en los features de día, mes y año. Además, en caso de representar categorías, de todos modos los atributos tienen un orden (las horas del día o los meses del año, por ejemplo). Por otro lado, uno solo de ellos es binario (si es día laborable o no). Incluso varios atributos representan variables reales, como la temperatura o la humedad. En el caso del atributo weatherseat, que tiene 4 categorías, se puede pensar en un "orden" porque un número más alto corresponde a una condición climática más extrema y por lo tanto, creemos, a menor uso de bicicletas.

Como casi todos los features pueden ser considerados como variables numéricas y debemos resolver un problema de regresión, lo abordaremos con una regresión lineal. Algunos atributos pueden ser suficientes para un esquema lineal (como el weathersit, ya que probablemente haya menos ciclistas cuanto más alto ese número); creemos que otros atributos tienen influencia "cuadrática" (pocos ciclistas en los extremos y máximo en algún lugar en el medio). Es el caso de aquellos atributos cuya falta o exceso puede desalentar el uso de bicicletas. Nos referimos, por ejemplo, al viento, la humedad, la temperatura, el mes del año (como vimos en los gráficos, hay más ciclistas en primavera y verano), o incluso la hora del día. Por eso, haremos una regresión lineal con los atributos y con el cuadrado de muchos de ellos, y todo esto por separado para predecir casual y registered.

Descartaremos el atributo fecha porque otros representan el mes y la hora, que estimamos más importantes; el season por ser prácticamente función del mes; el instant por ser un id de registro reemplazable por el índice de la tabla, la temp por ser similar a la atemp (y porque para salir a andar en bicicleta o no es más importante la sensación térmica que la temperatura). Tampoco usaremos el year porque queremos predecir el uso de bicicletas en un día determinado, el cual no debería depender de qué año es. Además, descartaremos el atributo weekday para reducir el riesgo de sobreajuste, y porque creemos que es mucho más importante si es día laborable o no respecto de si es lunes o jueves. No usaremos holiday porque creemos que tendrá una dinámica similar a un fin de semana, y esa información ya está en workingday. Como involucramos además el cuadrado de muchos de los features, usaremos regularización para reducir los parámetros y así prevenir el sobreajuste; probaremos las regularizaciones Ridge y Lasso, aunque creemos que a la primera le irá mejor. Esto se debe a que consideramos importantes a muchos atributos, y la regularización Lasso tiende a dar soluciones con muchos coeficientes muy pequeños o nulos.

Repasamos entonces las funciones de pérdida a utilizar para el desarrollo del modelo, donde MSE es el error cuadrático medio, w es el vector de coeficientes y w' es el vector de coeficientes sin el intercept:

L_ridge = MSE + ||w'||_2^2

L_lasso = MSE + ||w'||_1

En resumen, los atributos utilizados serán: month, month^2, hr, hr^2, workingday, weathersit, atemp, atemp^2, hum, hum^2, windspeed, windspeed^2. Para evaluar el desempeño de los modelos usaremos el score R^2.

Generamos ahora la nueva base de datos con todos los atributos a utilizar (originales y artificiales).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

In [8]:
from ucimlrepo import fetch_ucirepo
def get_data():
    ds = fetch_ucirepo(id=275)
    X_original = ds['data']['features'].copy()
    X_reg = ds['data']['features'].copy()
    target = ds['data']['original'][['casual','registered','cnt']]
    #X_reg['mnth'] = X_reg['mnth'].astype(float)
    #X_reg['hr'] = X_reg['hr'].astype(float)
    #X_reg['workingday'] = X_reg['workingday'].astype(float)
    X_reg['mnth^2'] = X_reg['mnth']**2
    X_reg['hr^2'] = X_reg['hr']**2
    X_reg['atemp^2'] = X_reg['atemp']**2
    X_reg['hum^2'] = X_reg['hum']**2
    X_reg['windspeed^2'] = X_reg['windspeed']**2
    X_reg = X_reg.drop(['dteday','yr','holiday','weekday','season','temp'],axis=1)
    return X_original, X_reg, target

In [9]:
X_original, X_reg, target = get_data()
#Convertimos a float casi todos los datos enteros, y luego los targets, para representarlos como variables reales en la función de pérdida.
#Dejamos weathersit como entero ya que lo usaremos para estratificar.

In [4]:
X_reg

,mnth,hr,workingday,weathersit,atemp,hum,windspeed,mnth^2,hr^2,atemp^2,hum^2,windspeed^2
0,1.0,0.0,0.0,1,0.2879,0.81,0.0000,1.0,0.0,0.082886,0.6561,0.000000
1,1.0,1.0,0.0,1,0.2727,0.80,0.0000,1.0,1.0,0.074365,0.6400,0.000000
2,1.0,2.0,0.0,1,0.2727,0.80,0.0000,1.0,4.0,0.074365,0.6400,0.000000
3,1.0,3.0,0.0,1,0.2879,0.75,0.0000,1.0,9.0,0.082886,0.5625,0.000000
4,1.0,4.0,0.0,1,0.2879,0.75,0.0000,1.0,16.0,0.082886,0.5625,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
17374,12.0,19.0,1.0,2,0.2576,0.60,0.1642,144.0,361.0,0.066358,0.3600,0.026962
17375,12.0,20.0,1.0,2,0.2576,0.60,0.1642,144.0,400.0,0.066358,0.3600,0.026962
17376,12.0,21.0,1.0,1,0.2576,0.60,0.1642,144.0,441.0,0.066358,0.3600,0.026962
17377,12.0,22.0,1.0,1,0.2727,0.56,0.1343,144.0,484.0,0.074365,0.3136,0.018036


In [17]:
target

,casual,registered,cnt
0,3,13,16
1,8,32,40
2,5,27,32
3,3,10,13
4,0,1,1
...,...,...,...
17374,11,108,119
17375,8,81,89
17376,7,83,90
17377,13,48,61


In [4]:
target_float = target.copy()
target_float['casual'] = target_float['casual'].astype(float)
target_float['registered'] = target_float['registered'].astype(float)
target_float['cnt'] = target_float['cnt'].astype(float)
target_float

,casual,registered,cnt
0,3.0,13.0,16.0
1,8.0,32.0,40.0
2,5.0,27.0,32.0
3,3.0,10.0,13.0
4,0.0,1.0,1.0
...,...,...,...
17374,11.0,108.0,119.0
17375,8.0,81.0,89.0
17376,7.0,83.0,90.0
17377,13.0,48.0,61.0


Es momento de dividir el dataset y el target en conjuntos de entrenamiento y test. Como tenemos más de 17.000 instancias, creemos que tomar el 10% para test nos dará suficientes casos de test. Por otro lado, dividiremos el conjunto restante en 5 folds. Recordamos que queremos que nuestro modelo aprenda a predecir el uso de bicicletas en un día cualquiera, con la información dada por los features que vimos. Por lo tanto, el conjunto de test y cada fold deberían evitar todo solapamiento de días, para evaluar cómo cada modelo entrenado con ciertos días rinde para otros. Como cada instancia (fila del dataset) corresponde a una fecha diferente, este objetivo se logra sin necesidad de agrupar. Además, no se requiere ninguna estratificación ya que no hay clases, sino un problema de regresión. Por otro lado, hemos visto que el weathersit, un atributo que consideramos de peso, tiene un notable desbalance de clases, y que el valor 4 (evento climático extremo) es estadísticamente insignificante. Queremos que nuestro modelo prediga bien la cantidad de ciclistas para weathersit = 1,2 o 3, así que haremos Stratified Group 5-Fold estratificando sobre este atributo. También cuidaremos las proporciones en el conjunto de test.

In [10]:
X_reg_train, X_reg_test, target_train, target_test = train_test_split(X_reg, target, test_size=0.1, random_state=42,
                                                                      stratify=X_reg['weathersit'])

In [46]:
X_reg_train.groupby(by='weathersit').count()

,mnth,hr,workingday,atemp,hum,windspeed,mnth^2,hr^2,atemp^2,hum^2,windspeed^2
weathersit,,,,,,,,,,,
1,10272,10272,10272,10272,10272,10272,10272,10272,10272,10272,10272
2,4089,4089,4089,4089,4089,4089,4089,4089,4089,4089,4089
3,1277,1277,1277,1277,1277,1277,1277,1277,1277,1277,1277
4,3,3,3,3,3,3,3,3,3,3,3


In [11]:
#Para poder realizar 5 folds, descartaremos las instancias con weathersit = 4 dado que sólo hay 3, lo que no permite dividir en 4 folds.
X_reg_train[X_reg_train['weathersit']==4] #los índices con este valor son el 585, el 8854 y el 9123.
mask = X_reg_train['weathersit'] != 4
X_reg_train = X_reg_train[mask]
target_train = target_train[mask][['casual','registered']]

In [28]:
def regresion_con_regularizacion(train_data,train_target,regularizacion):
    kfold = StratifiedKFold(n_splits=5)
    for alfa in [0.0,1.0,5.0,10.0,20.0,50.0,100.0,1000.0,10000.0]:
        model = Ridge(alpha=alfa,max_iter=10000)
        if regularizacion == 'lasso':
            model = Lasso(alpha=alfa,max_iter=200)
        r2_prom = np.zeros(2)
        for train_idx, val_idx in kfold.split(train_data, train_data['weathersit']):
            X_train_fold, X_val_fold = train_data.iloc[train_idx], train_data.iloc[val_idx]
            target_train_fold, target_val_fold = train_target.iloc[train_idx], train_target.iloc[val_idx]
            model.fit(X_train_fold,target_train_fold)
            pred_val_fold = model.predict(X_val_fold)
            mse = mean_squared_error(target_val_fold,pred_val_fold)
            #print(pred_val_fold-target_val_fold)
            r2_prom += np.array([r2_score(target_val_fold.iloc[:, 0], pred_val_fold[:, 0]),r2_score(target_val_fold.iloc[:, 1], pred_val_fold[:, 1])])
            print('MSE = ',mse)
        r2_prom = r2_prom/5
        print('score con Ridge para alpha = ',alfa,': ',r2_prom) 

In [15]:
regresion_con_regularizacion(X_reg_train,target_train,'ridge')

          casual  registered
16553  -5.426029   86.038004
8694  -32.915593  -27.603978
13823  61.381370   89.312671
8897    7.437616   72.437891
10817  -1.764952  -28.494669
...          ...         ...
13598 -42.690580 -303.461555
7863   12.226022 -136.688156
4708    5.748526   60.758806
9226  -22.389083 -387.130407
3819   -2.855620   44.012309

[3128 rows x 2 columns]
MSE =  7641.111549582031
          casual  registered
6462   30.339629   52.369956
6721   25.153142  160.323356
14900 -19.243567  -10.311619
7566    5.101714   87.434370
8051  -17.032859   34.380019
...          ...         ...
600   -23.089509   20.123531
8395   22.157051  161.131001
2604   15.273103   26.483874
8042   -4.656779 -102.040843
7338  -20.743577  -26.138501

[3128 rows x 2 columns]
MSE =  8151.346966437093
           casual  registered
12021  -14.384702   -2.263708
10845  -36.802126  -67.470678
1326    17.874861  123.589954
4315  -100.664601  -47.685752
1476     6.920466    5.305939
...           ...       

La regresión Ridge no se ve afectada por el parámetro alfa (ni por su ausencia), ya que siempre da lo mismo. Predice mejor el casual que el registered, pero de todos modos un R^2 que da aproximadamente 0.5 es malo. Además, como en 500 iteraciones no se observa una mejora respecto de si hay 200, concluimos que la regresión lineal con los atributos originales y artificiales que definimos no da buenos resultados.

In [ ]:
regresion_con_regularizacion(X_reg_train,target_train,'lasso')

c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.766e+05, tolerance: 3.057e+03
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.805e+06, tolerance: 2.927e+04
  model = cd_fast.enet_coordinate_descent(


MSE =  7778.171260345603


c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.856e+05, tolerance: 3.035e+03
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.698e+06, tolerance: 2.869e+04
  model = cd_fast.enet_coordinate_descent(


MSE =  8294.936459117744


c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.732e+05, tolerance: 2.996e+03
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.683e+06, tolerance: 2.866e+04
  model = cd_fast.enet_coordinate_descent(


MSE =  8234.790611021654


c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.818e+05, tolerance: 2.984e+03
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.800e+06, tolerance: 2.887e+04
  model = cd_fast.enet_coordinate_descent(


MSE =  8128.188275566267


c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.795e+05, tolerance: 2.996e+03
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.849e+06, tolerance: 2.902e+04
  model = cd_fast.enet_coordinate_descent(


MSE =  7916.206886223487
score con Lasso para alpha =  1.0 :  [0.45876974 0.35755159]
MSE =  8208.939294371963
MSE =  8817.265508397079
MSE =  8737.016283091247
MSE =  8623.83403239246
MSE =  8358.76115202595
score con Lasso para alpha =  5.0 :  [0.32845755 0.32974465]
MSE =  8616.591273938706
MSE =  9332.614406055949
MSE =  9198.872819380807
MSE =  9088.591213266038
MSE =  8784.903165406073
score con Lasso para alpha =  10.0 :  [0.21258833 0.30246979]
MSE =  9028.178453544097
MSE =  9802.438264293132
MSE =  9706.585400024565
MSE =  9537.187975569355
MSE =  9158.868816947332
score con Lasso para alpha =  20.0 :  [0.14590649 0.27118523]


Como esperábamos, a la regresión Lasso le va mucho peor, y además empeora cuanto más fuerte hacemos la regularización (y con alfa = 0 ni siquiera converge). El problema es que también le fue mal a la regresión Ridge.

Como en la regresión Ridge las predicciones de registered son mucho peores que las de casual, deducimos que podemos haber dejado de lado información importante sobre los usuarios registrados. Posiblemente a ellos les importen más algunos features que habíamos descartado, como el día de la semana o el holiday. Reentrenaremos las regresiones incorporando estos atributos, sin agregar otros artificiales ya que un usuario registered no necesariamente usará más la bicicleta en fines de semana que durante la semana hábil.

In [16]:
X_reg['weekday'] = X_original['weekday']
X_reg['holiday'] = X_original['holiday']

In [17]:
X_reg_train, X_reg_test, target_train, target_test = train_test_split(X_reg, target, test_size=0.1, random_state=42,
                                                                      stratify=X_reg['weathersit'])
X_reg_train[X_reg_train['weathersit']==4] #los índices con este valor son el 585, el 8854 y el 9123.
mask = X_reg_train['weathersit'] != 4
X_reg_train = X_reg_train[mask]
target_train = target_train[mask][['casual','registered']]

In [18]:
regresion_con_regularizacion(X_reg_train,target_train,'ridge')

          casual  registered
16553  -6.307390   84.278037
8694  -33.500271  -29.010450
13823  64.589347   94.638113
8897    7.682779   72.772418
10817   1.326762  -23.279287
...          ...         ...
13598 -42.935117 -303.844096
7863   11.680668 -137.945054
4708    5.811858   60.847077
9226  -22.128986 -386.784725
3819   -4.305405   41.146100

[3128 rows x 2 columns]
MSE =  7629.7008339857875
          casual  registered
6462   33.735942   56.588430
6721   25.095512  160.283294
14900 -20.266081  -11.734495
7566    5.168055   87.475156
8051  -16.858868   34.440932
...          ...         ...
600   -21.345579   22.570540
8395   22.556560  161.466980
2604   18.702992   30.825738
8042   -5.139442 -102.919710
7338  -22.370867  -28.515495

[3128 rows x 2 columns]
MSE =  8140.443094939485
           casual  registered
12021  -16.562774   -6.169869
10845  -38.940238  -71.353803
1326    16.081630  120.219965
4315  -102.426914  -52.543628
1476     4.824472    0.441871
...           ...      

Los resultados para Ridge son aproximadamente iguales, por lo que estos nuevos atributos no parecen tener mucha incidencia. No obstante, recordamos una observación que habíamos hecho en el archivo de exploración: hay muchos más usuarios registered en 2012 (y algunos más de tipo casual) que en 2011. Aunque el objetivo sigue siendo lograr predicciones independientes del año, podemos volver a correr estos modelos incorporando el año para ver si los resultados mejoran artificialmente. Otra alternativa es, ya que queremos predecir uso de bicicletas en el futuro, descartar directamente el año anterior ya que la realidad del último año debería ser más cercana a la del próximo. A continuación procedemos con estas dos alternativas.

In [29]:
X_reg['weekday'] = X_original['weekday']
X_reg['holiday'] = X_original['holiday']
X_reg['yr'] = X_original['yr']

In [30]:
X_reg_train, X_reg_test, target_train, target_test = train_test_split(X_reg, target, test_size=0.1, random_state=42,
                                                                      stratify=X_reg['weathersit'])
X_reg_train[X_reg_train['weathersit']==4] #los índices con este valor son el 585, el 8854 y el 9123.
mask = X_reg_train['weathersit'] != 4
X_reg_train = X_reg_train[mask]
target_train = target_train[mask][['casual','registered']]

In [21]:
X_reg

,mnth,hr,workingday,weathersit,atemp,hum,windspeed,mnth^2,hr^2,atemp^2,hum^2,windspeed^2,weekday,holiday,yr
0,1,0,0,1,0.2879,0.81,0.0000,1,0,0.082886,0.6561,0.000000,6,0,0
1,1,1,0,1,0.2727,0.80,0.0000,1,1,0.074365,0.6400,0.000000,6,0,0
2,1,2,0,1,0.2727,0.80,0.0000,1,4,0.074365,0.6400,0.000000,6,0,0
3,1,3,0,1,0.2879,0.75,0.0000,1,9,0.082886,0.5625,0.000000,6,0,0
4,1,4,0,1,0.2879,0.75,0.0000,1,16,0.082886,0.5625,0.000000,6,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17374,12,19,1,2,0.2576,0.60,0.1642,144,361,0.066358,0.3600,0.026962,1,0,1
17375,12,20,1,2,0.2576,0.60,0.1642,144,400,0.066358,0.3600,0.026962,1,0,1
17376,12,21,1,1,0.2576,0.60,0.1642,144,441,0.066358,0.3600,0.026962,1,0,1
17377,12,22,1,1,0.2727,0.56,0.1343,144,484,0.074365,0.3136,0.018036,1,0,1


In [31]:
regresion_con_regularizacion(X_reg_train,target_train,'ridge')

MSE =  9055.192815258388
MSE =  10080.431763707851
MSE =  10404.500663000437
MSE =  10563.016203803589
MSE =  9924.834450219736
score con Ridge para alpha =  0.0 :  [0.53592205 0.39913434]
MSE =  9069.683562350949
MSE =  10091.951745655693
MSE =  10401.955438275123
MSE =  10585.511729599848
MSE =  9923.33612755194
score con Ridge para alpha =  1.0 :  [0.5353743  0.39862082]
MSE =  9119.35489243638
MSE =  10137.773465042872
MSE =  10437.254607658831
MSE =  10648.272468809764
MSE =  9954.160614026934
score con Ridge para alpha =  5.0 :  [0.5323971  0.39602861]
MSE =  9144.963162818347
MSE =  10163.477018846781
MSE =  10458.92306134166
MSE =  10682.074874129457
MSE =  9971.588532769236
score con Ridge para alpha =  10.0 :  [0.53050178 0.39462249]
MSE =  9166.075334471794
MSE =  10187.277369088493
MSE =  10477.242105288355
MSE =  10714.739585767253
MSE =  9983.526332605381
score con Ridge para alpha =  20.0 :  [0.52831248 0.39346691]
MSE =  9192.663136965784
MSE =  10222.87509815822
MSE = 

Notamos que incorporar el año sí mejoró el R^2 para registered, pero no tanto como esperábamos. El siguiente paso es eliminar los registros del año 2011 para ver cuáles son los resultados con ese procedimiento.

In [32]:
X_original, X_reg, target = get_data()
X_reg['weekday'] = X_original['weekday']
X_reg['holiday'] = X_original['holiday']
X_reg['yr'] = X_original['yr']

mascara_clima = X_reg['weathersit'] != 4
X_reg = X_reg[mascara_clima]
target = target[mascara_clima][['casual','registered']]

mascara_2012 = X_reg['yr'] == 1
X_reg = X_reg[mascara_2012]
target = target[mascara_2012]

X_reg = X_reg.drop(['yr'],axis=1)

X_reg_train, X_reg_test, target_train, target_test = train_test_split(X_reg, target, test_size=0.1, random_state=42,
                                                                      stratify=X_reg['weathersit'])

In [33]:
regresion_con_regularizacion(X_reg_train,target_train,'ridge')

MSE =  9055.192815258375
MSE =  10080.43176370785
MSE =  10404.500663000417
MSE =  10563.016203803576
MSE =  9924.834450219725
score con Ridge para alpha =  0.0 :  [0.53592205 0.39913434]
MSE =  9069.683562350949
MSE =  10091.951745655693
MSE =  10401.955438275121
MSE =  10585.511729599848
MSE =  9923.33612755194
score con Ridge para alpha =  1.0 :  [0.5353743  0.39862082]
MSE =  9119.354892436379
MSE =  10137.773465042872
MSE =  10437.254607658831
MSE =  10648.272468809764
MSE =  9954.160614026932
score con Ridge para alpha =  5.0 :  [0.5323971  0.39602861]
MSE =  9144.963162818347
MSE =  10163.47701884678
MSE =  10458.92306134166
MSE =  10682.074874129456
MSE =  9971.588532769236
score con Ridge para alpha =  10.0 :  [0.53050178 0.39462249]
MSE =  9166.075334471794
MSE =  10187.277369088491
MSE =  10477.242105288355
MSE =  10714.739585767253
MSE =  9983.526332605381
score con Ridge para alpha =  20.0 :  [0.52831248 0.39346691]
MSE =  9192.663136965784
MSE =  10222.87509815822
MSE =  

In [18]:
regresion_con_regularizacion(X_reg_train,target_train,'lasso')

c:\Users\Lenny\Python39\lib\site-packages\sklearn\base.py:1474: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.745e+06, tolerance: 2.056e+03 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_d

MSE =  9055.490664039522
MSE =  10080.613054664567


c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.772e+07, tolerance: 1.933e+04 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\base.py:1474: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_d

MSE =  10403.517691088713
MSE =  10563.256352804046


c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.655e+06, tolerance: 2.017e+03 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features 

MSE =  9924.229267333274
score con Ridge para alpha =  0.0 :  [0.53593663 0.39914376]
MSE =  9231.0594917424


c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.173e+04, tolerance: 2.056e+03
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.089e+06, tolerance: 1.983e+04
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.031e+05, tolerance: 

MSE =  10283.495891950159
MSE =  10570.791835508693
MSE =  10854.152236462895


c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.616e+04, tolerance: 2.014e+03
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.598e+05, tolerance: 1.910e+04
  model = cd_fast.enet_coordinate_descent(
c:\Users\Lenny\Python39\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.011e+04, tolerance: 

MSE =  10008.848797062305
score con Ridge para alpha =  1.0 :  [0.49800697 0.39135065]
MSE =  9664.169572781278
MSE =  10805.731665718835
MSE =  11118.671848693437
MSE =  11506.951477479954
MSE =  10396.118128740636
score con Ridge para alpha =  5.0 :  [0.37217623 0.37191366]
MSE =  10110.919494437894
MSE =  11354.543817027878
MSE =  11674.958571631412
MSE =  12165.350858470649
MSE =  10820.867058500326
score con Ridge para alpha =  10.0 :  [0.25996137 0.34979195]
MSE =  10637.575839733667
MSE =  11955.180692930677
MSE =  12382.931497120955
MSE =  12907.56352132202
MSE =  11358.028928031015
score con Ridge para alpha =  20.0 :  [0.18801694 0.31716776]
MSE =  11361.958903028506
MSE =  12797.908533362859
MSE =  13330.243753107074
MSE =  13748.921435520342
MSE =  12131.45867228421
score con Ridge para alpha =  50.0 :  [0.04385347 0.27884798]
MSE =  12457.631617462199
MSE =  14116.379949255082
MSE =  14731.221327988866
MSE =  15022.46461988535
MSE =  13358.849841560024
score con Ridge para

Habiendo probado estas alternativas, vemos que la regresión lineal agregando variables cuadráticas no funciona, sobre todo teniendo en cuenta los MSE obtenidos, que son altísimos. Probaremos modelos de regresión con ensambles, comenzando por un RandomForestRegressor. Como la propia arquitectura de este modelo presenta aleatorización para prevenir el sobreajuste, usaremos todos los atributos de la base de datos excepto el instante (que funciona como id) y la fecha. Como incluimos el año, lo cual no es ideal, luego haremos lo mismo sólo para 2012.

In [46]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

In [37]:
X_original

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed
0,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0000
1,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0000
2,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0000
3,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0000
4,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
17374,2012-12-31,1,1,12,19,0,1,1,2,0.26,0.2576,0.60,0.1642
17375,2012-12-31,1,1,12,20,0,1,1,2,0.26,0.2576,0.60,0.1642
17376,2012-12-31,1,1,12,21,0,1,1,1,0.26,0.2576,0.60,0.1642
17377,2012-12-31,1,1,12,22,0,1,1,1,0.26,0.2727,0.56,0.1343


In [54]:
def regresion_random_forest(lista_max_depth,lista_min_samples_split,lista_max_features,train_data,train_target):
    kfold = StratifiedKFold(n_splits=5)
    """
    for alfa in [0.0,1.0,5.0,10.0,20.0,50.0,100.0,1000.0,10000.0]:
        model = Ridge(alpha=alfa,max_iter=10000)
        if regularizacion == 'lasso':
            model = Lasso(alpha=alfa,max_iter=200)
        r2_prom = np.zeros(2)
        for train_idx, val_idx in kfold.split(train_data, train_data['weathersit']):
            X_train_fold, X_val_fold = train_data.iloc[train_idx], train_data.iloc[val_idx]
            target_train_fold, target_val_fold = train_target.iloc[train_idx], train_target.iloc[val_idx]
            model.fit(X_train_fold,target_train_fold)
            pred_val_fold = model.predict(X_val_fold)
            mse = mean_squared_error(target_val_fold,pred_val_fold)
            #print(pred_val_fold-target_val_fold)
            r2_prom += np.array([r2_score(target_val_fold.iloc[:, 0], pred_val_fold[:, 0]),r2_score(target_val_fold.iloc[:, 1], pred_val_fold[:, 1])])
            print('MSE = ',mse)
        r2_prom = r2_prom/5
        print('score con Ridge para alpha = ',alfa,': ',r2_prom) 
    """
    for prof_max in lista_max_depth:
        for min_split in lista_min_samples_split:
            for max_feat in lista_max_features:
                model_1 = RandomForestRegressor(criterion='squared_error',max_depth=prof_max,min_samples_split=min_split,
                                              max_features=max_feat,random_state=42)
                model_2 = RandomForestRegressor(criterion='squared_error',max_depth=prof_max,min_samples_split=min_split,
                                                              max_features=max_feat,random_state=42)
                                
                r2_prom = np.zeros(2)
                for train_idx, val_idx in kfold.split(train_data, train_data['weathersit']):
                    X_train_fold, X_val_fold = train_data.iloc[train_idx], train_data.iloc[val_idx]
                    target_train_fold, target_val_fold = train_target.iloc[train_idx], train_target.iloc[val_idx]
                    model_1.fit(X_train_fold,target_train_fold.iloc[:,0])
                    pred_val_fold_1 = model_1.predict(X_val_fold)
                    model_2.fit(X_train_fold,target_train_fold.iloc[:,1])
                    pred_val_fold_2 = model_2.predict(X_val_fold)
                    r2_prom += np.array([r2_score(target_val_fold.iloc[:, 0], pred_val_fold_1),r2_score(target_val_fold.iloc[:, 1], pred_val_fold_2)])
                r2_prom = r2_prom/5
                print(prof_max,min_split,max_feat,'r2 promedio = ',r2_prom)

In [55]:
X_original, X_reg, target = get_data()
mascara_clima = X_original['weathersit'] != 4
X_original = X_original[mascara_clima]
target = target[mascara_clima]
X_reg = X_original.drop(['dteday'],axis=1)
X_reg_train, X_reg_test, target_train, target_test = train_test_split(X_reg, target, test_size=0.1, random_state=42,
                                                                      stratify=X_reg['weathersit'])
lista_max_depth = [3,5,7,None]
lista_min_samples_split = [2,5,25,50]
lista_max_features = ['sqrt',None]
regresion_random_forest(lista_max_depth,lista_min_samples_split,lista_max_features,X_reg_train,target_train)

3 2 sqrt r2 promedio =  [0.53329641 0.39905652]
3 2 None r2 promedio =  [0.58789582 0.45489322]
3 5 sqrt r2 promedio =  [0.53329641 0.39905652]
3 5 None r2 promedio =  [0.58789582 0.45489322]
3 25 sqrt r2 promedio =  [0.53329641 0.39905652]
3 25 None r2 promedio =  [0.58789582 0.45489322]
3 50 sqrt r2 promedio =  [0.53329153 0.39905652]
3 50 None r2 promedio =  [0.58789582 0.45489322]
5 2 sqrt r2 promedio =  [0.69444956 0.52322499]
5 2 None r2 promedio =  [0.77247277 0.62429365]
5 5 sqrt r2 promedio =  [0.6936929  0.52335205]
5 5 None r2 promedio =  [0.77247277 0.62429371]
5 25 sqrt r2 promedio =  [0.69219864 0.52345259]
5 25 None r2 promedio =  [0.77245777 0.62429341]
5 50 sqrt r2 promedio =  [0.69146845 0.52295853]
5 50 None r2 promedio =  [0.77240197 0.62429408]
7 2 sqrt r2 promedio =  [0.78456107 0.64357441]
7 2 None r2 promedio =  [0.83663788 0.84992678]
7 5 sqrt r2 promedio =  [0.78945583 0.64496131]
7 5 None r2 promedio =  [0.83662001 0.8499208 ]
7 25 sqrt r2 promedio =  [0.7819

In [49]:
def regresion_hist_boosting(lista_max_depth,lista_min_samples_leaf,lista_max_features,train_data,train_target):
    kfold = StratifiedKFold(n_splits=5)
    """
    for alfa in [0.0,1.0,5.0,10.0,20.0,50.0,100.0,1000.0,10000.0]:
        model = Ridge(alpha=alfa,max_iter=10000)
        if regularizacion == 'lasso':
            model = Lasso(alpha=alfa,max_iter=200)
        r2_prom = np.zeros(2)
        for train_idx, val_idx in kfold.split(train_data, train_data['weathersit']):
            X_train_fold, X_val_fold = train_data.iloc[train_idx], train_data.iloc[val_idx]
            target_train_fold, target_val_fold = train_target.iloc[train_idx], train_target.iloc[val_idx]
            model.fit(X_train_fold,target_train_fold)
            pred_val_fold = model.predict(X_val_fold)
            mse = mean_squared_error(target_val_fold,pred_val_fold)
            #print(pred_val_fold-target_val_fold)
            r2_prom += np.array([r2_score(target_val_fold.iloc[:, 0], pred_val_fold[:, 0]),r2_score(target_val_fold.iloc[:, 1], pred_val_fold[:, 1])])
            print('MSE = ',mse)
        r2_prom = r2_prom/5
        print('score con Ridge para alpha = ',alfa,': ',r2_prom) 
    """
    for prof_max in lista_max_depth:
        for min_leaf in lista_min_samples_leaf:
            for max_feat in lista_max_features:
                model_1 = HistGradientBoostingRegressor(loss='squared_error',max_depth=prof_max,min_samples_leaf=min_leaf,
                                              max_features=max_feat,random_state=42)
                model_2 = HistGradientBoostingRegressor(loss='squared_error',max_depth=prof_max,min_samples_leaf=min_leaf,
                                                              max_features=max_feat,random_state=42)
                r2_prom = np.zeros(2)
                for train_idx, val_idx in kfold.split(train_data, train_data['weathersit']):
                    X_train_fold, X_val_fold = train_data.iloc[train_idx], train_data.iloc[val_idx]
                    target_train_fold, target_val_fold = train_target.iloc[train_idx], train_target.iloc[val_idx]
                    model_1.fit(X_train_fold,target_train_fold.iloc[:,0])
                    model_2.fit(X_train_fold,target_train_fold.iloc[:,1])
                    pred_val_fold_1 = model_1.predict(X_val_fold)
                    pred_val_fold_2 = model_2.predict(X_val_fold)
                    r2_prom += np.array([r2_score(target_val_fold.iloc[:, 0], pred_val_fold_1),r2_score(target_val_fold.iloc[:, 1], pred_val_fold_2)])
                r2_prom = r2_prom/5
                print(prof_max,min_leaf,max_feat,'r2 promedio = ',r2_prom)

In [57]:
X_original, X_reg, target = get_data()
mascara_clima = X_original['weathersit'] != 4
X_original = X_original[mascara_clima]
target = target[mascara_clima]
X_reg = X_original.drop(['dteday'],axis=1)
X_reg_train, X_reg_test, target_train, target_test = train_test_split(X_reg, target, test_size=0.1, random_state=42,
                                                                      stratify=X_reg['weathersit'])
lista_max_depth = [3,5,7,None]
lista_min_samples_leaf = [1,5,25,50]
lista_max_features = [0.25,0.5,0.75,1.0]
regresion_hist_boosting(lista_max_depth,lista_min_samples_leaf,lista_max_features,X_reg_train,target_train)

3 1 0.25 r2 promedio =  [0.80726725 0.75332276]
3 1 0.5 r2 promedio =  [0.82549591 0.84356925]
3 1 0.75 r2 promedio =  [0.82482373 0.84317821]
3 1 1.0 r2 promedio =  [0.82618613 0.8556242 ]
3 5 0.25 r2 promedio =  [0.80461726 0.75447179]
3 5 0.5 r2 promedio =  [0.82491951 0.84130427]
3 5 0.75 r2 promedio =  [0.8251372  0.84651456]
3 5 1.0 r2 promedio =  [0.82600858 0.85661067]
3 25 0.25 r2 promedio =  [0.80748336 0.74343426]
3 25 0.5 r2 promedio =  [0.82517226 0.84738976]
3 25 0.75 r2 promedio =  [0.82469699 0.84549799]
3 25 1.0 r2 promedio =  [0.82554084 0.85745944]
3 50 0.25 r2 promedio =  [0.80724498 0.74283777]
3 50 0.5 r2 promedio =  [0.82299898 0.84004979]
3 50 0.75 r2 promedio =  [0.82486471 0.83601426]
3 50 1.0 r2 promedio =  [0.82620751 0.85653417]
5 1 0.25 r2 promedio =  [0.88179673 0.8791599 ]
5 1 0.5 r2 promedio =  [0.89235188 0.9316863 ]
5 1 0.75 r2 promedio =  [0.89397742 0.93432361]
5 1 1.0 r2 promedio =  [0.89301776 0.93523803]
5 5 0.25 r2 promedio =  [0.88225704 0.8781

Ahora repetimos el experimento para ambos modelos tomando sólo el año 2012.

In [56]:
X_original, X_reg, target = get_data()
mascara_clima = X_original['weathersit'] != 4
X_original = X_original[mascara_clima]
target = target[mascara_clima]
mascara_2012 = X_original['yr'] == 1
X_original = X_original[mascara_2012]
target = target[mascara_2012]
X_reg = X_original.drop(['dteday'],axis=1)
X_reg_train, X_reg_test, target_train, target_test = train_test_split(X_reg, target, test_size=0.1, random_state=42,
                                                                      stratify=X_reg['weathersit'])
lista_max_depth = [3,5,7,None]
lista_min_samples_split = [2,5,25,50]
lista_max_features = ['sqrt',None]
regresion_random_forest(lista_max_depth,lista_min_samples_split,lista_max_features,X_reg_train,target_train)

3 2 sqrt r2 promedio =  [0.56144173 0.39320557]
3 2 None r2 promedio =  [0.61055489 0.49033031]
3 5 sqrt r2 promedio =  [0.56144173 0.39320557]
3 5 None r2 promedio =  [0.61055489 0.49033031]
3 25 sqrt r2 promedio =  [0.56144173 0.39304784]
3 25 None r2 promedio =  [0.61055489 0.49033031]
3 50 sqrt r2 promedio =  [0.56122967 0.39233422]
3 50 None r2 promedio =  [0.61055489 0.49033031]
5 2 sqrt r2 promedio =  [0.71842236 0.52862143]
5 2 None r2 promedio =  [0.80371167 0.77339144]
5 5 sqrt r2 promedio =  [0.71682417 0.5304172 ]
5 5 None r2 promedio =  [0.80370919 0.77339284]
5 25 sqrt r2 promedio =  [0.72154269 0.52675866]
5 25 None r2 promedio =  [0.80370249 0.77339659]
5 50 sqrt r2 promedio =  [0.71945193 0.5360724 ]
5 50 None r2 promedio =  [0.80375154 0.77336719]
7 2 sqrt r2 promedio =  [0.81130487 0.66023885]
7 2 None r2 promedio =  [0.85589007 0.88490568]
7 5 sqrt r2 promedio =  [0.80565287 0.65095154]
7 5 None r2 promedio =  [0.85587374 0.88491413]
7 25 sqrt r2 promedio =  [0.8050

In [58]:
X_original, X_reg, target = get_data()
mascara_clima = X_original['weathersit'] != 4
X_original = X_original[mascara_clima]
target = target[mascara_clima]
mascara_2012 = X_original['yr'] == 1
X_original = X_original[mascara_2012]
target = target[mascara_2012]
X_reg = X_original.drop(['dteday'],axis=1)
X_reg_train, X_reg_test, target_train, target_test = train_test_split(X_reg, target, test_size=0.1, random_state=42,
                                                                      stratify=X_reg['weathersit'])
lista_max_depth = [3,5,7,None]
lista_min_samples_leaf = [1,5,25,50]
lista_max_features = [0.25,0.5,0.75,1.0]
regresion_hist_boosting(lista_max_depth,lista_min_samples_leaf,lista_max_features,X_reg_train,target_train)

3 1 0.25 r2 promedio =  [0.83138551 0.75357682]
3 1 0.5 r2 promedio =  [0.84949426 0.86742961]
3 1 0.75 r2 promedio =  [0.8498726  0.86685619]
3 1 1.0 r2 promedio =  [0.84963113 0.87926312]
3 5 0.25 r2 promedio =  [0.83112521 0.7481359 ]
3 5 0.5 r2 promedio =  [0.84917827 0.87216396]
3 5 0.75 r2 promedio =  [0.84832725 0.86996284]
3 5 1.0 r2 promedio =  [0.85076391 0.88134541]
3 25 0.25 r2 promedio =  [0.82969928 0.74705616]
3 25 0.5 r2 promedio =  [0.84526787 0.8702491 ]
3 25 0.75 r2 promedio =  [0.84493629 0.86856378]
3 25 1.0 r2 promedio =  [0.8471917  0.87872047]
3 50 0.25 r2 promedio =  [0.82781365 0.74586737]
3 50 0.5 r2 promedio =  [0.84361856 0.86728616]
3 50 0.75 r2 promedio =  [0.84377862 0.87125027]
3 50 1.0 r2 promedio =  [0.84640367 0.88162299]
5 1 0.25 r2 promedio =  [0.89999959 0.89628508]
5 1 0.5 r2 promedio =  [0.90999978 0.93784545]
5 1 0.75 r2 promedio =  [0.91035759 0.94151278]
5 1 1.0 r2 promedio =  [0.91037661 0.94146305]
5 5 0.25 r2 promedio =  [0.898969   0.8934

Observamos que la calidad de ambos modelos va mejorando a medida que se les permite más profundidad a los árboles, y que el HistGradientBoostingRegresor es superior al RandomForestRegresor. Elegiremos nuestro modelo definitivo entre los mejores del boosting, que tienen R^2 mayor a 0.9 tanto para casual como para registered, lo que consideramos un excelente resultado.

Aunque los valores más altos se encuentran cuando los árboles no tienen profundidad limitada (max_depth = None), elegiremos un modelo entre los de max_depth = 7, ya que también logran muy buenos resultados y creemos que van a sobreajustar menos. Como queremos predicciones para años futuros, elegimos un modelo a partir de los resultados para el caso donde se excluyó el año 2011. Concretamente, tomaremos el HistGradientBoostingRegresor con max_depth = 7, min_samples_split = 1 y max_features = 1.0, cuyos R^2 scores son: [0.92098183 0.94844226] (para casual y registered respectivamente).

Ya con este modelo elegido, vamos a evaluar su resultado con los datos de evaluación, y luego construiremos las predicciones de cnt sumando las de casual y registered. Luego de aplicar la suma redondeamos ya que se debe predecir valores enteros (uso de bicicletas en el día).

In [68]:
X_original, X_reg, target = get_data()
mascara_clima = X_original['weathersit'] != 4
X_original = X_original[mascara_clima]
target = target[mascara_clima]
mascara_2012 = X_original['yr'] == 1
X_original = X_original[mascara_2012]
target = target[mascara_2012]
X_reg = X_original.drop(['dteday'],axis=1)
X_reg_train, X_reg_test, target_train, target_test = train_test_split(X_reg, target, test_size=0.1, random_state=42,
                                                                      stratify=X_reg['weathersit'])

In [69]:
modelo_final_1 = HistGradientBoostingRegressor(max_depth=7,min_samples_leaf=1,max_features=1.0,loss='squared_error')
modelo_final_2 = HistGradientBoostingRegressor(max_depth=7,min_samples_leaf=1,max_features=1.0,loss='squared_error')
modelo_final_1.fit(X_reg_train,target_train['casual'])
modelo_final_2.fit(X_reg_train,target_train['registered'])
pred_casual = modelo_final_1.predict(X_reg_test)
pred_registered = modelo_final_2.predict(X_reg_test)
pred_cnt = np.round(pred_casual+pred_registered,0)

In [70]:
pred_cnt

array([ 19., 141., 173., 384.,  14., 279.,  21.,   0., 672., 285.,  37.,
        60., 607., 490., 647.,  27.,  19.,  71., 499.,  47., 198., 293.,
       192., 129., 208., 469.,  99., 535., 472.,  52., 120., 205.,  56.,
       305., 313., 389., 219., 305., 188., 456., 126., 189.,  96., 897.,
       194.,  29., 513., -13., 195., 351., 764., 190., 281., 242.,  18.,
        70., 419., 394.,  12.,  53.,  11., 632., 396.,  73., 338.,  13.,
        14.,   3., 233., 436.,  13., 284.,  12.,  17., 701., 499., 227.,
       340., 155., 618., 421., 615., 270., 575., 167., 511., -12.,  30.,
       196.,  50., 483.,  27., 207., 184., 465., 509., 438., 134.,  73.,
       528.,  13., 117., 316., 215.,  40., 442., 130., 278., 201., 514.,
       166., 128., 114., 113., 180., 432.,  50.,  18., 218.,  91., 513.,
       217., 305.,  28., 433.,  42., 168.,  42., 228., 139.,   7., 574.,
       217., 556., 228., 365., 403., 295., 175., 152.,  14., 434.,  44.,
       688., 380., 342.,  96.,  82.,  11., 214.,   

In [71]:
target_test

,casual,registered,cnt
14781,0,9,9
17275,23,151,174
9711,13,111,124
13159,124,218,342
10681,0,4,4
...,...,...,...
16213,1,106,107
14455,73,221,294
16809,3,9,12
16318,144,325,469


In [72]:
r2_score(target_test.iloc[:,0],pred_casual)

0.9134097229047091

In [73]:
r2_score(target_test.iloc[:,1],pred_registered)

0.9569765130369485

In [74]:
mean_squared_error(target_test.iloc[:,0],pred_casual)

276.8215591921389

In [76]:
mean_squared_error(target_test.iloc[:,1],pred_registered)

1304.554230969874

In [62]:
print(pred_cnt-np.array(target_test.iloc[:,2]))

[  10.  -33.   49.   42.   10.   11.    2.   -2.    8.  -15.  -18.   28.
   11.  374.    7.  -11.   -9.    5.   83.   13.  108.  -38.   62.   13.
    5.  -37.    7.   19.  -83.   31.   90.   11.    2.  -12.  -22.   48.
  -15.  -73.   27. -111.   27.   -2.   73.  -41.   22.   19.  -50.  -22.
  -24.  -22.   12.   33.   21.  -25.  -16.   17.   11.  -51.    7.    0.
   -9. -125.  -28.   19.   65.    7.    2.   -1.    9. -148.    1.   21.
    6.    7.   -1.   35.  -36.   36.    1.   50.    3.  -73.  -40.   50.
   13. -103.  -19.  -16.  -25.   36.   64.  -16.   -2.  -19.  -65.  -13.
  -33.   30.    7.  -11.    3.   -6.  -18.   27.   14.  -30.   16.  -51.
    0.  -67.  -24.  -20.   27.   13.    8.  132.   18.   16.  -66.    3.
   30.   36.  -37.  -26.   22.  -13.   -3.   13.   49. -105.    3.  -88.
  -28.  -89.   24.   62.  -40.   80.   -7.  -15.  -22.  -41.    8.   14.
 -111.   -9.  -17.  -23.    2.    3.    7.  -97.   -3.  114.  -69.   53.
  -21.    9.   32.  -10.   19.   51.   59.   48.   

In [ ]:
print((pred_cnt-np.array(target_test.iloc[:,2]))/np.array(target_test.iloc[:,2]))
#calculamos los errores relativos

[ 1.11111111e+00 -1.89655172e-01  3.95161290e-01  1.22807018e-01
  2.50000000e+00  4.10447761e-02  1.05263158e-01 -1.00000000e+00
  1.20481928e-02 -5.00000000e-02 -3.27272727e-01  8.75000000e-01
  1.84563758e-02  3.22413793e+00  1.09375000e-02 -2.89473684e-01
 -3.21428571e-01  7.57575758e-02  1.99519231e-01  3.82352941e-01
  1.20000000e+00 -1.14803625e-01  4.76923077e-01  1.12068966e-01
  2.46305419e-02 -7.31225296e-02  7.60869565e-02  3.68217054e-02
 -1.49549550e-01  1.47619048e+00  3.00000000e+00  5.67010309e-02
  3.70370370e-02 -3.78548896e-02 -6.56716418e-02  1.40762463e-01
 -6.41025641e-02 -1.93121693e-01  1.67701863e-01 -1.95767196e-01
  2.72727273e-01 -1.04712042e-02  3.17391304e+00 -4.37100213e-02
  1.27906977e-01  1.90000000e+00 -8.88099467e-02 -2.44444444e+00
 -1.09589041e-01 -5.89812332e-02  1.59574468e-02  2.10191083e-01
  8.07692308e-02 -9.36329588e-02 -4.70588235e-01  3.20754717e-01
  2.69607843e-02 -1.14606742e-01  1.40000000e+00  0.00000000e+00
 -4.50000000e-01 -1.65125

Al menos en muchos casos hemos conseguido errores en cnt menores al 10% (los de orden e-2, e-3 o menores), aunque en otros hemos cometido errores mayores. Llegamos a la conclusión de que es un resultado aceptable para un problema que intrínsecamente es ruidoso por depender de la decisión de mucha gente de subirse o no a una bicicleta, aunque la confianza que se puede tener en nuestro modelo por sí solo es limitada. Hemos logrado muy buenos valores de R^2 como son 0,91 para casual y 0,96 para registered, lo que al menos indica que nuestro modelo es muy superior al predictor "dummy" que sería predecir el promedio de uso diario para cualquier día.